In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

df = pd.read_csv('student_grade_ses_dataset.csv')

# Provide idea of the dataframe we are dealing with 
print(df.info())
print(df.describe())

In [ ]:
# Data Preprocessing 

# 1. Cleaning Duplicate Rows 
duplicates = df[df.duplicated()]
print("Duplicate Rows:")
print(duplicates)

row_count_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
row_count_after = len(df)
print(f"{row_count_before} rows -> {row_count_after} after removing duplicates")

# 2. Set 'student_id' as the index and drop it from the standard columns
is_all_unique = df['student_id'].is_unique
print(f"Column student_id uniqueness: {is_all_unique}")
df = df.set_index('student_id', drop=True)


## TODO: What to do about missing rows  

In [ ]:
# Check for missing values 
missing_rows = df[df.isna().any(axis=1)]
print("Rows with missing values:")
print(missing_rows) 

# Calculate percentage of missing cells per column 
col_missing_pct = df.isna().mean() * 100

# Filter to only show columns that actually have missing data, sorted highest to lowest
print("\nPercentage of missing values per column:")
print(col_missing_pct[col_missing_pct > 0].sort_values(ascending=False))

# TEMPORARY: drop rows with missing values until a proper imputation strategy is added
rows_before = len(df)
df = df.dropna()
print(f"\n{rows_before} rows -> {len(df)} after dropping rows with missing values")

In [ ]:
# Visualisation of predictors 

# Columns with only a few integer values get one bar per value
# Default histogram bins cut across the integers and leave gaps
num_cols = df.select_dtypes('number').columns

fig, axes = plt.subplots(4, 4, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    if df[col].nunique() <= 10:
        df[col].value_counts().sort_index().plot(kind='bar', ax=ax)
        ax.set_xlabel('')
    else:
        df[col].hist(bins=30, ax=ax)
    ax.set_title(col, fontsize=9)

plt.tight_layout()
plt.show()

## Skewed income column: annual_family_income_inr

From the histogram above, we can see that the annual_family_income_inr column is very right-skewed. In the data it ranges from about ₹60,000 to ₹19.6 million, a factor of more than 300. Most families sit near the low end and a few earn far more. On the raw scale, the few very rich families stretch the axis. This creates the problems of the ultra-wealthy outliers having too much influence in linear models. 

In such cases, a log transformation is ideal because algorithms like linear regression, logistics regression, K-means clustering and PCA are highly sensitive to extreme values and perform better when features are normally distributed. As the log is a monotonically increasing function, it naturally squashes massive numbers closer together while preserving the exact rank order of data.

In [ ]:
# From the histogram above, heavily skewed income, so log-transform 
print(f"annual_family_income_inr skew: {df['annual_family_income_inr'].skew()}")
df['log_income'] = np.log(df.pop('annual_family_income_inr'))
print(f"log_income skew: {df['log_income'].skew()}")

## Schema Setup

Based off the variable histograms above, we can deduce which of the columns are discrete/continuous/categorical and we can form the column schema below. 

# TODO: CONSIDER CATEGORIAL PREDICTION

In [ ]:
# Schema Setup 

TARGET_COL = 'final_grade_pct' 

CONTINUOUS_COLS = [
    'distance_to_school_km', 
    'study_hours_per_week', 
    'attendance_rate_pct',          
    'previous_year_grade_pct', 
    'log_income'
]

# Integer-coded, including 0/1 flags and the 1-5 parental involvement score
DISCRETE_COLS = [
    'age', 
    'grade_level', 
    'family_size', 
    'num_siblings', 
    'single_parent_household',
    'has_internet_access', 
    'has_computer_or_laptop', 
    'extra_tutoring',
    'parental_involvement_score'
]

# Text labels
CATEGORICAL_COLS = [
    'academic_year', 
    'gender', 
    'school_type', 
    'residence_type', 
    'family_income_bracket',
    'father_education_level', 
    'mother_education_level', 
    'father_occupation',
    'mother_occupation', 
    'scholarship_status'
]


# Derived columns are computed from final_grade_pct
# grade_class  -> final_grade_pct cut into bands (F < 40, D 40-59.9, C 60-74.9, B 75-89.9, A >= 90)
# at_risk_flag -> 1 exactly when final_grade_pct < 40
DERIVED_COLS = ['grade_class', 'at_risk_flag']

# survey_date is not a predictor: all surveys were taken in March-May 2025
NOT_USED = ['survey_date']

NUMERIC_COLS = CONTINUOUS_COLS + DISCRETE_COLS
PREDICTOR_COLS = NUMERIC_COLS + CATEGORICAL_COLS

# check the target-derived columns really are bands of the target, and that none ended up as predictors
print(df.groupby('grade_class')[TARGET_COL].agg(['min', 'max']))
print(df.groupby('at_risk_flag')[TARGET_COL].agg(['min', 'max']))
assert not set(DERIVED_COLS + NOT_USED) & set(PREDICTOR_COLS)

In [ ]:
X = df[PREDICTOR_COLS]
Y = df[TARGET_COL]
print('X:', X.shape, ' Y:', Y.shape)

## Highly Correlated Predictors 



In [ ]:
# Correlation heatmap of the numeric predictors
# (categorical columns are text, so they are left out here)
corr = X[NUMERIC_COLS].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)

# write the correlation value in each square
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, round(corr.iloc[i, j], 2), ha='center', va='center', fontsize=7)

plt.title('Correlation between numeric predictors')
plt.tight_layout()
plt.show()

From the heatmap, most predictors are only weakly correlated with each other. Two pairs stand out:

- `age` and `grade_level` (r = 0.93): students in the same grade are mostly the same age.
- `family_size` and `num_siblings` (r = 0.94): more siblings means a bigger family.

Keeping both columns of a pair gives the linear models the same information twice, which makes the OLS coefficients unstable. So we drop one column from each pair:

- Keep `grade_level`, drop `age`. Grade level tells us which curriculum the student is in, and age within a grade adds little.
- Keep `family_size`, drop `num_siblings`. Family size also counts parents and other relatives, so it is the more general measure.

Both columns in each pair are almost uncorrelated with `final_grade_pct` (|r| about 0.00 for age/grade level and 0.05 for family size/siblings), so the choice is made on what the variable means rather than on the target. This also avoids using the target before the train/test split.

In [ ]:
# Drop one predictor from each highly correlated pair
# age vs grade_level (r = 0.93) -> keep grade_level
# family_size vs num_siblings (r = 0.94) -> keep family_size
DROP_COLS = ['age', 'num_siblings']

DISCRETE_COLS = [c for c in DISCRETE_COLS if c not in DROP_COLS]
NUMERIC_COLS = CONTINUOUS_COLS + DISCRETE_COLS
PREDICTOR_COLS = NUMERIC_COLS + CATEGORICAL_COLS

X = df[PREDICTOR_COLS]
print('Dropped:', DROP_COLS)
print('X:', X.shape)

In [ ]:
# Setup helper functions for data preparation so that X-train and X-test can be massaged into a format palatable to each model 

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, SplineTransformer
from sklearn.base import BaseEstimator, RegressorMixin
from pygam import LinearGAM, s, l      # pip install pygam
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Columns with no natural order: one-hot encoded
ONEHOT_COLUMNS = ['gender', 'school_type', 'residence_type', 'academic_year',
                  'father_occupation', 'mother_occupation']

# Columns with a clear order (lowest level first): ordinal encoded
ORDINAL_COLUMNS = {
    'father_education_level': [
        'No Formal Education', 'Primary', 'Secondary', 'Higher Secondary',
        "Bachelor's Degree", "Master's Degree", 'Doctorate',
    ],
    'mother_education_level': [
        'No Formal Education', 'Primary', 'Secondary', 'Higher Secondary',
        "Bachelor's Degree", "Master's Degree", 'Doctorate',
    ],
    'family_income_bracket': ['Low', 'Lower-Middle', 'Middle', 'Upper-Middle', 'High'],
    'scholarship_status': ['No Scholarship', 'Partial', 'Full'],
}

def build_prep(scale):
    # one-hot and ordinal encoders are the same for every family. Only scaling differs
    return ColumnTransformer([
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ONEHOT_COLUMNS),
        ('ordinal', OrdinalEncoder(categories=list(ORDINAL_COLUMNS.values())), list(ORDINAL_COLUMNS)),
        ('num', StandardScaler() if scale else 'passthrough', NUMERIC_COLS),
    ])

def make_linear_prep():  # OLS, Ridge, Lasso
    return build_prep(scale=True)

def make_spline_prep():  # Natural cubic spline regression: splines on the continuous columns only
    return ColumnTransformer([
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ONEHOT_COLUMNS),
        ('ordinal', OrdinalEncoder(categories=list(ORDINAL_COLUMNS.values())), list(ORDINAL_COLUMNS)),
        ('continuous', SplineTransformer(n_knots=5, degree=3, extrapolation='linear'), CONTINUOUS_COLS),
        ('discrete', StandardScaler(), DISCRETE_COLS),
    ])

def make_gam_prep():     # GAM: splines handle the shape, so no scaling
    return build_prep(scale=False)

class GAMRegressor(BaseEstimator, RegressorMixin):
    # Wrapper so pygam's LinearGAM works inside a Pipeline and GridSearchCV
    # Smooth spline term for each continuous column, linear term for every other column
    def __init__(self, n_splines=10, lam=0.6):
        self.n_splines = n_splines
        self.lam = lam      # smoothing penalty, bigger = smoother curves

    def fit(self, X, y):
        first_continuous = X.shape[1] - len(NUMERIC_COLS)   # one-hot and ordinal columns come first
        last_continuous = first_continuous + len(CONTINUOUS_COLS)
        terms = l(0)
        for j in range(1, X.shape[1]):
            if first_continuous <= j < last_continuous:
                terms = terms + s(j, n_splines=self.n_splines, lam=self.lam)
            else:
                terms = terms + l(j)
        self.gam_ = LinearGAM(terms).fit(X, y)
        return self

    def predict(self, X):
        return self.gam_.predict(X)

def make_knn_prep():     # KNN: distances need scaled columns
    return build_prep(scale=True)

def make_tree_prep():    # decision tree, random forest
    return build_prep(scale=False)

def make_gbt_prep():     # gradient boosting
    return build_prep(scale=False)

def make_svr_prep():     # SVR: RBF kernel uses distances, so scale
    return build_prep(scale=True)

def make_pca_prep():     # PCA: variance depends on scale, so scale first
    return build_prep(scale=True)

def compute_test_metrics(fitted_model, X_test, Y_test):
    # Returns (RMSE, MAE, R2) of a fitted model on the test set
    test_predictions = fitted_model.predict(X_test)
    return (root_mean_squared_error(Y_test, test_predictions),
            mean_absolute_error(Y_test, test_predictions),
            r2_score(Y_test, test_predictions))



In [ ]:
# Pilot: find sensible hyperparameter ranges
# One wide grid search per model on split 0 (training rows only), then keep the values around the best one
# The test set is never touched here

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

X_pilot, _, Y_pilot, _ = train_test_split(X, Y, test_size=0.15, random_state=0, shuffle=True)
pilot_cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Wide grids
wide_ridge_grid         = {'model__alpha': [0.1, 1, 3, 10, 30, 100, 300, 1000, 3000]}
wide_lasso_grid         = {'model__alpha': [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]}
wide_spline_grid        = {'prep__continuous__n_knots': [3, 4, 5, 6, 8],
                           'model__alpha':              [1, 10, 100, 1000]}
wide_gam_grid           = {'model__lam': [0.1, 1, 10, 100, 1000]}
wide_knn_grid           = {'model__n_neighbors': [5, 10, 25, 50, 100, 200, 400],
                           'model__weights':     ['uniform', 'distance']}
wide_decision_tree_grid = {'model__max_depth':        [2, 3, 4, 6, 8, 10, 15],
                           'model__min_samples_leaf': [5, 10, 20, 50, 100, 200, 400]}
wide_random_forest_grid = {'model__max_features':     [0.1, 0.2, 0.33, 0.5, 0.8, 1.0],
                           'model__min_samples_leaf': [1, 5, 10, 20, 40, 80]}
wide_gbt_grid           = {'model__learning_rate':    [0.01, 0.02, 0.05, 0.1, 0.2, 0.4],
                           'model__max_depth':        [1, 2, 3, 4, 6],
                           'model__min_samples_leaf': [10, 20, 50, 100, 200]}
wide_svr_grid           = {'model__C':     [0.1, 1, 3, 10, 30, 100],
                           'model__gamma': [0.001, 0.003, 0.01, 0.03]}
wide_pca_grid           = {'pca__n_components': [1, 2, 5, 10, 15, 20, 25, 30, 38]}

# Name: (pipeline, wide grid)
pilot_models = {
    'Ridge':          (Pipeline([('prep', make_linear_prep()), ('model', Ridge())]),                                                              wide_ridge_grid),
    'Lasso':          (Pipeline([('prep', make_linear_prep()), ('model', Lasso(max_iter=10000))]),                                                wide_lasso_grid),
    'Spline':         (Pipeline([('prep', make_spline_prep()), ('scale', StandardScaler()), ('model', Ridge())]),                                 wide_spline_grid),
    'GAM':            (Pipeline([('prep', make_gam_prep()),    ('model', GAMRegressor())]),                                                       wide_gam_grid),
    'k-NN':           (Pipeline([('prep', make_knn_prep()),    ('model', KNeighborsRegressor())]),                                                wide_knn_grid),
    'Decision tree':  (Pipeline([('prep', make_tree_prep()),   ('model', DecisionTreeRegressor(random_state=0))]),                                wide_decision_tree_grid),
    'Random forest':  (Pipeline([('prep', make_tree_prep()),   ('model', RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1))]),   wide_random_forest_grid),
    'GBT':            (Pipeline([('prep', make_gbt_prep()),    ('model', HistGradientBoostingRegressor(max_iter=500, early_stopping=True, random_state=0))]), wide_gbt_grid),
    'SVR':            (Pipeline([('prep', make_svr_prep()),    ('model', SVR(kernel='rbf'))]),                                                    wide_svr_grid),
    'PCA regression': (Pipeline([('prep', make_pca_prep()),    ('pca', PCA()), ('model', LinearRegression())]),                                   wide_pca_grid),
}

def narrow(values, best_value):
    # keep the best value and one neighbour on each side
    i = values.index(best_value)
    return values[max(i - 1, 0): i + 2]

pilot_results = {}
best_values   = {}
final_grids   = {}

for name, (pipe, grid) in pilot_models.items():
    search = GridSearchCV(pipe, grid, cv=pilot_cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    search.fit(X_pilot, Y_pilot)
    pilot_results[name] = pd.DataFrame(search.cv_results_)
    best_values[name]   = search.best_params_
    final_grids[name]   = {}

    print(f'{name}: best CV RMSE = {-search.best_score_:.3f}')
    for param, values in grid.items():
        best = search.best_params_[param]
        if isinstance(values[0], str):      # text options (weights) are kept as they are
            final_grids[name][param] = values
        else:
            final_grids[name][param] = narrow(values, best)
        edge = '  <-- on the edge of the wide grid' if best in (values[0], values[-1]) and not isinstance(best, str) else ''
        print(f'    {param}: best = {best}, final grid = {final_grids[name][param]}{edge}')

# Plot CV RMSE for each numeric hyperparameter (best score over the other hyperparameters)
# red line = best value, green band = range passed to the main script
for name, (pipe, grid) in pilot_models.items():
    params = [p for p in grid if not isinstance(grid[p][0], str)]
    fig, axes = plt.subplots(1, len(params), figsize=(5 * len(params), 3.5), squeeze=False)
    for ax, param in zip(axes[0], params):
        curve = -pilot_results[name].groupby('param_' + param)['mean_test_score'].max()
        ax.plot(curve.index.astype(float), curve.values, marker='o')
        ax.axvspan(min(final_grids[name][param]), max(final_grids[name][param]), color='green', alpha=0.2)
        ax.axvline(best_values[name][param], color='red', linestyle='--')
        if max(grid[param]) / min(grid[param]) > 20:
            ax.set_xscale('log')
        ax.set_xlabel(param.split('__')[-1])
        ax.set_ylabel('CV RMSE')
        ax.set_title(name)
    plt.tight_layout()
    plt.show()

# Pass the final grids on to the main script below
ridge_grid         = final_grids['Ridge']
lasso_grid         = final_grids['Lasso']
spline_grid        = final_grids['Spline']
gam_grid           = final_grids['GAM']
knn_grid           = final_grids['k-NN']
decision_tree_grid = final_grids['Decision tree']
random_forest_grid = final_grids['Random forest']
gbt_grid           = final_grids['GBT']
svr_grid           = final_grids['SVR']
pca_grid           = final_grids['PCA regression']



In [ ]:
# Main Script 

# Imports 
import time

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA

# Store (RMSE, MAE, R2) on the test set for each iteration, one tuple per loop 
errors_naive                = []
errors_ols                  = []
errors_ridge                = []
errors_lasso                = []
errors_spline_regression    = []
errors_gam                  = []
errors_k_nearest_neighbors  = []
errors_decision_tree        = []
errors_random_forest        = []
errors_gradient_boosting    = []
errors_svr                  = []
errors_pca_regression       = []


# CONST HELPERS
TEST_SIZE              = 0.15 # 85/15 split 
NUMBER_OF_EXPERIMENTS  = 20   # Each experiment = new random 85/15 split, tune every model, score on the test set. 20 runs for rigorous statistical difference analysis

# HYPERPARAMETERS: ridge_grid, lasso_grid, spline_grid, gam_grid, knn_grid, decision_tree_grid, random_forest_grid,
# gbt_grid, svr_grid and pca_grid
# come from the pilot cell above (run it first). Keys use the pipeline step prefix: model__ or pca__

for experiment in range(NUMBER_OF_EXPERIMENTS): 
    
    # Step 1. 85/15 train/test split: standardized baseline for model comparison across models for this experiment 
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=TEST_SIZE, random_state=experiment, shuffle=True)
    experiment_start_time = time.time()
    print('train:', X_train.shape, ' test:', X_test.shape)
    
    # Step 2. Model-specific pre-processing (one-hot/ordinal encoding, standardization etc.)
    # Each pipeline chains a preprocessor with a final model, so it can be fitted and used to predict directly

    naive_pipe               = Pipeline([('prep', make_linear_prep()), ('model', DummyRegressor(strategy='mean'))])                                                             # Linear: naive mean
    ols_pipe                 = Pipeline([('prep', make_linear_prep()), ('model', LinearRegression())])                                                                          # Linear: OLS
    ridge_pipe               = Pipeline([('prep', make_linear_prep()), ('model', Ridge())])                                                                                     # Linear: Ridge
    lasso_pipe               = Pipeline([('prep', make_linear_prep()), ('model', Lasso(max_iter=10000))])                                                                       # Linear: Lasso
    spline_regression_pipe   = Pipeline([('prep', make_spline_prep()), ('scale', StandardScaler()), ('model', Ridge())])                                                        # Natural cubic spline regression
    gam_pipe                 = Pipeline([('prep', make_gam_prep()),    ('model', GAMRegressor())])                                                                              # GAM
    k_nearest_neighbors_pipe = Pipeline([('prep', make_knn_prep()),    ('model', KNeighborsRegressor())])                                                                       # k-NN
    decision_tree_pipe       = Pipeline([('prep', make_tree_prep()),   ('model', DecisionTreeRegressor(random_state=experiment))])                                              # Decision tree
    random_forest_pipe       = Pipeline([('prep', make_tree_prep()),   ('model', RandomForestRegressor(n_estimators=100, random_state=experiment, n_jobs=-1))])                 # Random forest
    gradient_boosting_pipe   = Pipeline([('prep', make_gbt_prep()),    ('model', HistGradientBoostingRegressor(max_iter=500, early_stopping=True, random_state=experiment))])   # Boosting / GBT
    svr_pipe                 = Pipeline([('prep', make_svr_prep()),    ('model', SVR(kernel='rbf'))])                                                                           # SVR (RBF kernel)
    pca_regression_pipe      = Pipeline([('prep', make_pca_prep()),    ('pca', PCA()), ('model', LinearRegression())])                                                          # PCA regression

    # Step 3. GridSearchCV - K-folds on 85% training data with aim of finding the best hyperparameters.
    cv = KFold(n_splits=10, shuffle=True, random_state=experiment)
    # Random forest is the slowest model, so it uses 5 folds instead of 10
    random_forest_cv = KFold(n_splits=5, shuffle=True, random_state=experiment)
    scoring = 'neg_root_mean_squared_error'

    # Define the grid searches
    ridge_search                = GridSearchCV(ridge_pipe, ridge_grid, cv=cv, scoring=scoring)
    lasso_search                = GridSearchCV(lasso_pipe, lasso_grid, cv=cv, scoring=scoring)
    spline_regression_search    = GridSearchCV(spline_regression_pipe, spline_grid, cv=cv, scoring=scoring, n_jobs=-1)
    gam_search                  = GridSearchCV(gam_pipe, gam_grid, cv=cv, scoring=scoring, n_jobs=-1)
    k_nearest_neighbors_search  = GridSearchCV(k_nearest_neighbors_pipe, knn_grid, cv=cv, scoring=scoring, n_jobs=-1)
    decision_tree_search        = GridSearchCV(decision_tree_pipe, decision_tree_grid, cv=cv, scoring=scoring, n_jobs=-1)
    random_forest_search        = GridSearchCV(random_forest_pipe, random_forest_grid, cv=random_forest_cv, scoring=scoring)
    gradient_boosting_search    = GridSearchCV(gradient_boosting_pipe, gbt_grid, cv=cv, scoring=scoring)
    svr_search                  = GridSearchCV(svr_pipe, svr_grid, cv=random_forest_cv, scoring=scoring, n_jobs=-1)   # 5 folds, SVR is slow
    pca_regression_search       = GridSearchCV(pca_regression_pipe, pca_grid, cv=cv, scoring=scoring, n_jobs=-1)

    # Fit everything on the 85% training data
    naive_pipe.fit(X_train, Y_train)      # predicts the training mean, no hyperparameters
    ols_pipe.fit(X_train, Y_train)        # OLS has no hyperparameters
    ridge_search.fit(X_train, Y_train)
    lasso_search.fit(X_train, Y_train)
    spline_regression_search.fit(X_train, Y_train)
    gam_search.fit(X_train, Y_train)
    k_nearest_neighbors_search.fit(X_train, Y_train)
    decision_tree_search.fit(X_train, Y_train)
    random_forest_search.fit(X_train, Y_train)
    gradient_boosting_search.fit(X_train, Y_train)
    svr_search.fit(X_train, Y_train)
    pca_regression_search.fit(X_train, Y_train)
    
    # Step 4. Evaluate best chosen hyperparameter for each model and validate using 15% testing data set 
    # GridSearchCV objects predict with their best refitted pipeline
    fitted_models = [
        (naive_pipe, errors_naive),
        (ols_pipe, errors_ols),
        (ridge_search, errors_ridge),
        (lasso_search, errors_lasso),
        (spline_regression_search, errors_spline_regression),
        (gam_search, errors_gam),
        (k_nearest_neighbors_search, errors_k_nearest_neighbors),
        (decision_tree_search, errors_decision_tree),
        (random_forest_search, errors_random_forest),
        (gradient_boosting_search, errors_gradient_boosting),
        (svr_search, errors_svr),
        (pca_regression_search, errors_pca_regression)
    ]
    
    # Step 5. Record performance metrics (RMSE, MAE, R2) for this iteration 
    for fitted_model, error_list in fitted_models:
        error_list.append(compute_test_metrics(fitted_model, X_test, Y_test))
    print(f'experiment {experiment + 1}/{NUMBER_OF_EXPERIMENTS} done in {time.time() - experiment_start_time:.0f}s')
    
    # Step 6. Create some visualisations if necessary. 
    
    


## Model Performance Metrics 

For each model, we have used RMSE, MAE and R2-score. 

TODO: Also explore using MLE as performance metric (week 2)



## Visualisation: Box plots and confidence intervals

In [ ]:
names = ['Naive', 'OLS', 'Ridge', 'Lasso', 'Spline', 'GAM', 'k-NN', 'Decision tree', 'Random forest', 'GBT', 'SVR', 'PCA']
model_error_lists = [errors_naive, errors_ols, errors_ridge, errors_lasso, errors_spline_regression, errors_gam,
                     errors_k_nearest_neighbors, errors_decision_tree, errors_random_forest, errors_gradient_boosting,
                     errors_svr, errors_pca_regression]

# each loop stores (RMSE, MAE, R2), so column 0, 1, 2 is one measure
for col, metric in enumerate(['RMSE', 'MAE', 'R2']):
    plt.figure(figsize=(9, 5))
    plt.boxplot([np.array(e)[:, col] for e in model_error_lists])
    plt.xticks(range(1, len(names) + 1), names)
    plt.ylabel(metric)
    plt.title(f'{metric} on the test set ({NUMBER_OF_EXPERIMENTS} experiments)')
    plt.show()
